In [ ]:
!pip install datasets
from datasets import load_dataset

# Load the dataset
dataset_dict = load_dataset("cnn_dailymail", "3.0.0")

# Convert the 'train', 'validation', and 'test' splits to pandas DataFrames
train_df = dataset_dict['train'].to_pandas()
val_df = dataset_dict['validation'].to_pandas()
test_df = dataset_dict['test'].to_pandas()


train_df = dataset_dict['train'].select(range(1000))
val_df =  dataset_dict['validation'].select(range(200))
test_df =  dataset_dict['test'].select(range(200))

# Print the shapes of the DataFrames
print("Training data shape:", train_df.shape)
print("Validation data shape:", val_df.shape)
print("Test data shape:", test_df.shape)


Training data shape: (1000, 3)
Validation data shape: (200, 3)
Test data shape: (200, 3)


In [ ]:
!pip install keras tensorflow
import tensorflow as tf
from tensorflow.keras.layers import Layer, Embedding, Bidirectional, LSTM, Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

In [ ]:
class WordAttention(Layer):
    def __init__(self, **kwargs):
        super(WordAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="attention_weight",
                                 shape=(input_shape[-1], input_shape[-1]),
                                 initializer="random_normal",
                                 trainable=True)
        self.b = self.add_weight(name="attention_bias",
                                 shape=(input_shape[-1],),
                                 initializer="zeros",
                                 trainable=True)
        self.u = self.add_weight(name="attention_context_vector",
                                 shape=(input_shape[-1], 1),
                                 initializer="random_normal",
                                 trainable=True)
        super(WordAttention, self).build(input_shape)

    def call(self, x):
        x = tf.cast(x, tf.float32)
        print("Input shape to WordAttention:", x.shape)

        # Get dynamic shape values
        batch_size = tf.shape(x)[0]
        num_tokens = tf.shape(x)[1]  # Get num_tokens dynamically from input shape
        feature_dim = x.shape[-1]  # Use the last dimension as feature_dim

        #Calculate uit
        uit = K.tanh(K.dot(x, self.W) + self.b)
        print("uit shape:", uit.shape)

        #Calculate ait
        ait = K.dot(uit, self.u)
        print("ait shape:", ait.shape)

        #Apply softmax to get attention weights
        ait = tf.nn.softmax(ait, axis=1)
        print("ait shape after softmax:", ait.shape)

        #Perform element-wise multiplication
        weighted_input = x * ait # (batch_size, num_tokens, feature_dim)
        print("weighted_input shape:", weighted_input.shape)

        #Sum weighted inputs to get context vector
        output = K.sum(weighted_input, axis=1) # (batch_size, feature_dim)
        print("output shape:", output.shape)

        return output

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

In [ ]:
class SentenceAttention(tf.keras.layers.Layer):
    def __init__(self, attention_dim):
        super(SentenceAttention, self).__init__()
        self.attention_dim = attention_dim
        self.W = tf.keras.layers.Dense(attention_dim)
        self.u = tf.keras.layers.Dense(1)

    def call(self, x):
        # x shape: (batch_size, sentence_count, embedding_dim)

        # Generate uit
        uit = self.W(x)  # (batch_size, sentence_count, attention_dim)

        # Generate attention weights
        ait = self.u(uit)  # (batch_size, sentence_count, 1)
        ait = tf.nn.softmax(ait, axis=1)

        # Ensure proper broadcasting
        weighted_input = x * ait  # Broadcasting should work now

        # Sum over the sentence dimension
        output = tf.reduce_sum(weighted_input, axis=1)  # (batch_size, embedding_dim)

        return output

In [ ]:
def call(self, x):
    print(f"SentenceAttention input shape: {x.shape}")
    uit = self.W(x)
    print(f"uit shape: {uit.shape}")
    ait = self.u(uit)
    print(f"ait shape before softmax: {ait.shape}")
    ait = tf.nn.softmax(ait, axis=1)
    print(f"ait shape after softmax: {ait.shape}")
    weighted_input = x * ait
    print(f"weighted_input shape: {weighted_input.shape}")
    output = tf.reduce_sum(weighted_input, axis=1)
    print(f"output shape: {output.shape}")
    return output

In [ ]:
from datasets import DatasetDict

# Define dataset_dict using DatasetDict
dataset_dict = DatasetDict({
    "train": train_df,
    "validation": val_df,
    "test": test_df
})

# Verify the dataset
print(dataset_dict)


DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 200
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 200
    })
})


In [ ]:
from transformers import PegasusTokenizer, PegasusForConditionalGeneration

from collections import defaultdict
import string
from string import punctuation
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import PegasusConfig
import re
import math
import torch.nn.functional as F

class PegasusMemoryConfig(PegasusConfig):
    def __init__(self, n_gram_size=3, max_memory_size=10000, **kwargs):
        super().__init__(**kwargs)
        self.n_gram_size = n_gram_size
        self.max_memory_size = max_memory_size

class PegasusWithMemory(PegasusForConditionalGeneration):
    @classmethod
    def from_pretrained(cls, pretrained_model_name_or_path, *model_args, **kwargs):
        # First load the base model configuration
        config = kwargs.pop('config', None)
        if config is None:
            config = PegasusMemoryConfig.from_pretrained(pretrained_model_name_or_path)

        # Ensure memory-specific parameters are set
        if not hasattr(config, 'n_gram_size'):
            config.n_gram_size = 3
        if not hasattr(config, 'max_memory_size'):
            config.max_memory_size = 10000

        # Initialize model with the configuration
        model = super().from_pretrained(
            pretrained_model_name_or_path,
            config=config,
            *model_args,
            **kwargs
        )

        # Initialize memory-specific attributes
        model.memory = {}
        model.tokenizer = PegasusTokenizer.from_pretrained(pretrained_model_name_or_path)

        return model

    #N-Grams

    def remove_punctuation(self, text):
        if(type(text)==float):
            return text
        ans=""
        for i in text:
            if i not in string.punctuation:
                ans+=i
        return ans

    def generate_n_grams(self, text, n = 3):
        text = self.remove_punctuation(text)
        words = [word for word in text.split(" ") if word not in set(stopwords.words('english'))]
        n_grams = zip(*[words[i:] for i in range(n)])
        return [' '.join(ngram) for ngram in n_grams]

    def update_memory(self, text, input_data):
        ngrams = self.generate_n_grams(text)
        for ngram in ngrams:
            if ngram not in self.memory:
                self.memory[ngram] = set()
            self.memory[ngram].add(input_data)

            if len(self.memory[ngram]) > self.config.max_memory_size:
                list(self.memory[ngram]).pop(0)

    #TF-IDF

    def prepare_retrieval_data(self, corpus, query, top_k=2, threshold=0.6):
        if isinstance(corpus[0], list):
            corpus = [" ".join(doc) for doc in corpus]

        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(corpus + [query])
        query_vector = tfidf_matrix[-1]

        similarities = cosine_similarity(query_vector, tfidf_matrix[:-1]).flatten()

        # Filter for indices where the similarity is above the threshold.
        valid_indices = [i for i, sim in enumerate(similarities) if sim > threshold]

        # If no document meets the threshold, return an empty list.
        if not valid_indices:
            return []

        # Sort the valid indices in descending order of similarity.
        valid_indices = sorted(valid_indices, key=lambda i: similarities[i], reverse=True)

        # Select the top_k indices from the sorted valid indices.
        top_k_indices = valid_indices[:top_k]

        # Return the corresponding documents from the corpus.
        return [corpus[idx] for idx in top_k_indices]

    def retrieve_memory(self, text):
        ngrams = self.generate_n_grams(text)
        relevant_data = []
        output = []
        for ngram in ngrams:
            if ngram in self.memory:
                relevant_data.extend(self.memory[ngram])
                output = self.prepare_retrieval_data(list(set(relevant_data)), text)

        return output

    def apply_rope(self, q, k):
        seq_len, d = q.shape[-2], q.shape[-1]
        theta = 10000 ** (-2 * (torch.arange(0, d, 2).float() / d))
        theta = theta.to(q.device)

        positions = torch.arange(seq_len, device=q.device).float()
        theta = torch.outer(positions, theta)

        cos_theta = torch.cos(theta)
        sin_theta = torch.sin(theta)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rotated = torch.cat([q1 * cos_theta - q2 * sin_theta, q1 * sin_theta + q2 * cos_theta], dim=-1)
        k_rotated = torch.cat([k1 * cos_theta - k2 * sin_theta, k1 * sin_theta + k2 * cos_theta], dim=-1)

        return q_rotated, k_rotated

    def forward(self, input_ids=None,  decoder_input_ids=None, attention_mask=None, labels=None, num_of_items=None, num_items_in_batch=None, **kwargs):
        if input_ids is not None:
            # Get original text
            text = self.tokenizer.batch_decode(input_ids, skip_special_tokens=True)[0]

            # Update memory by storing the whole summary
            if labels != None:
                data = ""
                 # Loop through each sequence in the batch
                for label_seq in labels:
                    # Convert the tensor to a list of token IDs
                    token_ids = label_seq.tolist()

                    # Filter out the ignore index (-100)
                    filtered_ids = [token for token in token_ids if token != -100]

                    # Decode the filtered token IDs to text
                    decoded_text = tokenizer.decode(filtered_ids, skip_special_tokens=True)
                    data += decoded_text

                self.update_memory(text, data)

            encoder_outputs = self.model.encoder(input_ids, attention_mask=attention_mask, return_dict=True)
            hidden_states = encoder_outputs.last_hidden_state

            # Access the first layer of the encoder
            first_layer = self.model.encoder.layers[0]

            # Extract query and key projections
            q = first_layer.self_attn.q_proj(hidden_states)
            k = first_layer.self_attn.k_proj(hidden_states)

            # Apply RoPE to the projections
            q_rope, k_rope = self.apply_rope(q, k)

            # Compute attention scores with RoPE-transformed queries and keys
            attention_scores = torch.matmul(q_rope, k_rope.transpose(-1, -2)) / math.sqrt(q.size(-1))
            attention_probs = F.softmax(attention_scores, dim=-1)

            # Compute context layer
            context_layer = torch.matmul(attention_probs, first_layer.self_attn.v_proj(hidden_states))
            hidden_states = first_layer.self_attn.out_proj(context_layer)

            # *Fix: Ensure compatibility with generate()*
            outputs = super().forward(
                input_ids=input_ids,
                attention_mask=attention_mask,
                decoder_input_ids=decoder_input_ids,
                labels=labels,
                return_dict=True,
                **kwargs
            )
            return outputs

        return super().forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            labels=labels,
            **kwargs
        )

    def generate(self, input_ids=None, attention_mask=None, **kwargs):
        if input_ids is not None:
            text = self.tokenizer.batch_decode(input_ids, skip_special_tokens=True)[0]
            memory_data = self.retrieve_memory(text)

            if memory_data:
                augmented_text = memory_data[0] + "\n" + text
                # augmented_text = text + " " + memory_data[0] --> For News-Sum Dataset

                inputs = self.tokenizer(
                    augmented_text,
                    truncation=True,
                    padding=True,
                    max_length=self.config.max_position_embeddings,
                    return_tensors="pt"
                ).to(input_ids.device)

                input_ids = inputs["input_ids"]
                attention_mask = inputs["attention_mask"]

        # Update memory: Storing the summary generated by the model

        output = super().generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs
        )

        data = self.tokenizer.decode(output[0], skip_special_tokens=True)
        self.update_memory(text, data)

        return output



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
Some weights of PegasusWithMemory were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM
from tensorflow.keras.models import Model,Sequential

def hierarchical_attention_model(vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix):
    sentence_input = Input(shape=(sentence_count, word_count))

    # Word encoder
    word_encoder = Sequential([
        Embedding(vocab_size, embedding_dim, weights=[embedding_matrix], trainable=False),
        Bidirectional(LSTM(100, return_sequences=True)),
        WordAttention()
    ])

    # Apply word encoder to each sentence
    sentence_encoder = tf.keras.layers.TimeDistributed(word_encoder)(sentence_input)
    sentence_bi_lstm = Bidirectional(LSTM(100, return_sequences=True))(sentence_encoder)
    # Pass attention_dim when creating SentenceAttention
    sentence_attention = SentenceAttention(attention_dim=200)(sentence_bi_lstm)

    return tf.keras.models.Model(inputs=sentence_input, outputs=sentence_attention)

In [ ]:
import numpy as np
vocab_size = 10000
embedding_dim = 300
sentence_count = 10
word_count = 20
embedding_matrix = np.random.rand(vocab_size, embedding_dim)

hierarchical_model = hierarchical_attention_model(vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix)

In [ ]:
hierarchical_model.compile()

In [ ]:
hierarchical_model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)           │ (None, 10, 20)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_2 (TimeDistributed) │ (None, 10, 200)             │       3,361,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_5 (Bidirectional)      │ (None, 10, 200)             │         240,800 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ sentence_attention_2                 │ (None, 200)                 │          40,401 │
│ (SentenceAttention)                  │                             │                 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,642,401 (13.89 MB)

 Trainable params: 642,401 (2.45 MB)

 Non-trainable params: 3,000,000 (11.44 MB)

In [ ]:
import torch

In [ ]:
def HTA(batch, tokenizer, sentence_count, word_count):
    # Tokenize input articles
    inputs = tokenizer(
        batch["article"],
        truncation=True,
        padding="max_length",
        max_length=min(512, sentence_count * word_count),
        return_tensors="np"
    )

    # # Reshape input to fit HAN
    # input_ids = np.array(inputs["input_ids"]).reshape(-1, sentence_count, word_count)

    # # Convert token IDs to embeddings using HAN (Optional if you're not using embeddings directly)
    # hta_output = hierarchical_model(input_ids)

    # # Tokenize target summaries (highlights)
    targets = tokenizer(
        batch["highlights"],
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    return {
        "input_ids": inputs["input_ids"],
        "labels": targets["input_ids"]
    }



In [ ]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

def prepare_dataset(dataset_dict, vocab_size, embedding_dim, sentence_count, word_count, tokenizer, embedding_matrix, hierarchical_attention_model):
    # Define preprocessing function with fixed parameters
    hta_model_instance = hierarchical_attention_model(vocab_size, embedding_dim, sentence_count, word_count, embedding_matrix)
    def preprocess_with_fixed_params(batch):
        return HTA(
            batch,
            tokenizer,
            sentence_count,
            word_count
        )


    processed_datasets = {}
    for split in dataset_dict.keys():
        processed_datasets[split] = dataset_dict[split].map(
            preprocess_with_fixed_params,
            batched=True,
            batch_size=8,
            remove_columns=dataset_dict[split].column_names,
            desc=f"Processing {split} split"
        )

    return processed_datasets

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
processed_dataset_dict = prepare_dataset(
    dataset_dict=dataset_dict,
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    sentence_count=sentence_count,
    word_count=word_count,
    tokenizer=tokenizer,
    embedding_matrix=embedding_matrix,
    hierarchical_attention_model=hierarchical_attention_model
)

Processing train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Processing validation split:   0%|          | 0/200 [00:00<?, ? examples/s]

Processing test split:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
print(processed_dataset_dict["train"])
print(processed_dataset_dict["validation"])
print(processed_dataset_dict["test"])

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 1000
})
Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 200
})
Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 200
})


In [ ]:
print(processed_dataset_dict)

{'train': Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 1000
}), 'validation': Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 200
}), 'test': Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 200
})}


In [ ]:
from transformers import PegasusTokenizer, PegasusForConditionalGeneration, Trainer, TrainingArguments, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./pegasus-finetuned",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=7,
    weight_decay=0.001,
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    predict_with_generate=True,
    remove_unused_columns=False,
)


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset_dict["train"],
    eval_dataset=processed_dataset_dict["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

<ipython-input-56-0ef18ffda59f>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,5.518300,0.701686
2,4.654000,0.709995
3,4.052600,0.721477
4,3.715800,0.725202
5,3.495600,0.731877
6,3.668800,0.738871
7,3.208200,0.741560


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=875, training_loss=4.011631267002651, metrics={'train_runtime': 3330.4378, 'train_samples_per_second': 2.102, 'train_steps_per_second': 0.263, 'total_flos': 3950439628800000.0, 'train_loss': 4.011631267002651, 'epoch': 7.0})

In [ ]:
def evaluate_summaries(dataset, model, tokenizer):
    references = []
    predictions = []

    for sample in dataset:
        input_text = f"{sample['article']} {sample['highlights']}"
        reference_summary = sample['highlights']
        references.append(reference_summary)

        # Generate summary
        inputs = tokenizer(input_text, max_length=1024, truncation=True, return_tensors="pt").to(model.device)
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=128,
            num_beams=4,
            early_stopping=True
        )
        generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        predictions.append(generated_summary)

    return references, predictions


In [ ]:
test_references, test_predictions = evaluate_summaries(test_df, model, tokenizer)

In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score

def calculate_average_precision(test_references, test_predictions):
    # Create a set of all unique words across human and predicted summaries
    all_words = set(word for summary in test_references for word in summary.lower().split())
                # set(word for summary in test_predictions for word in summary.lower().split())

    # Convert human summaries and predicted summaries into binary vectors
    human_vectors = [
        np.array([1 if word in summary.lower().split() else 0 for word in all_words])
        for summary in test_references
    ]
    predicted_vectors = [
        np.array([1 if word in summary.lower().split() else 0 for word in all_words])
        for summary in test_predictions
    ]

    # Calculate the average precision score
    ap_scores = []
    for true, pred in zip(human_vectors, predicted_vectors):
        try:
            ap_scores.append(average_precision_score(true, pred))
        except ValueError:
            ap_scores.append(0.0)

    return np.mean(ap_scores)

In [ ]:
ap_score = calculate_average_precision(test_references, test_predictions)
print(f"Average Precision Score: {ap_score}")

In [ ]:
!pip install evaluate
!pip install rouge_score
import evaluate
rouge = evaluate.load("rouge")

In [ ]:
results = rouge.compute(predictions=test_predictions, references=test_references)

In [ ]:
print(f"ROUGE-1: {results['rouge1']}")
print(f"ROUGE-2: {results['rouge2']}")
print(f"ROUGE-L: {results['rougeL']}")

ROUGE-1: 0.43477382939239034
ROUGE-2: 0.2535883710658394
ROUGE-L: 0.34505704931413816


In [ ]:
index = 0
print("Reference Summary:")
print(test_references[index])
print("\nPredicted Summary:")
print(test_predictions[index])

Reference Summary:
Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .

Predicted Summary:
The Palestinian Authority formally becomes the 123rd member of the International Criminal Court . Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories . Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


In [ ]:
import pandas as pd

# Create a DataFrame
df_results = pd.DataFrame({
    "Reference Summary": test_references,
    "Predicted Summary": test_predictions
})

# Save to CSV file
df_results.to_csv("summary_comparison.csv", index=False)

print("CSV file saved successfully!")


CSV file saved successfully!


In [ ]:
from nltk.translate.bleu_score import corpus_bleu

# Tokenize references and predictions
tokenized_references = [[ref.split()] for ref in test_references]  # BLEU expects a list of lists
tokenized_predictions = [pred.split() for pred in test_predictions]

# Compute BLEU Score
bleu_score = corpus_bleu(tokenized_references, tokenized_predictions)

print(f"BLEU Score: {bleu_score:.4f}")


BLEU Score: 0.2061


In [ ]:
from nltk.translate.bleu_score import corpus_bleu

# Tokenize references and predictions
tokenized_references = [[ref.split()] for ref in test_references]  # BLEU expects a list of lists
tokenized_predictions = [pred.split() for pred in test_predictions]

# Compute BLEU Score
bleu_score = corpus_bleu(tokenized_references, tokenized_predictions)

print(f"BLEU Score: {bleu_score:.4f}")


BLEU Score: 0.2061


In [ ]:
import nltk
nltk.download('punkt')

from nltk.translate.bleu_score import sentence_bleu, corpus_bleu

# Compute BLEU for each sentence
for i in range(2):  # Print BLEU scores for the first 5 samples
    reference = [test_references[i].split()]  # BLEU expects a list of lists
    prediction = test_predictions[i].split()

    score = sentence_bleu(reference, prediction)
    print(f"Sample {i+1} - BLEU Score: {score:.4f}")


Sample 1 - BLEU Score: 0.6577
Sample 2 - BLEU Score: 0.3312


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
!pip install bert_score
from bert_score import score

# Assuming test_predictions contains the generated summaries
generated_summaries = test_predictions

# Assuming test_references contains the actual summaries
actual_summaries = test_references

P, R, F1 = score(generated_summaries, actual_summaries, lang="en", model_type="roberta-large")

avg_precision = P.mean().item()
bert_f1_score = F1.mean().item()

print(f"\n🔹 BERTScore (Improved): {bert_f1_score:.4f}")
print(f" Average Precision (BERTScore P): {avg_precision:.4f}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🔹 BERTScore (Improved): 0.8922
 Average Precision (BERTScore P): 0.8896
